In [1]:
import importlib
from pathlib import Path
import sys

# Mount Google Drive if running in Google Colab
try:
    from google.colab import drive
    drive.mount("/content/drive")
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = "https://github.com/pratik0620/adaptive-federated-aggregation.git"

if IN_COLAB or Path("/content").exists():
    REPO_ROOT = Path("/content/adaptive-federated-healthcare")
    # If alternate clone directory exists and primary does not, use it
    if not REPO_ROOT.exists():
        alt_root = Path("/content/adaptive-federated-aggregation")
        if alt_root.exists() and (alt_root / "src").is_dir():
            REPO_ROOT = alt_root
        else:
            !git clone {REPO_URL} {REPO_ROOT}
    
    # If repository already exists, safely sync with remote and discard local dirty diffs
    if REPO_ROOT.exists() and (REPO_ROOT / ".git").is_dir():
        !git -C {REPO_ROOT} fetch origin main
        !git -C {REPO_ROOT} reset --hard origin/main
else:
    REPO_ROOT = Path.cwd()
    if not (REPO_ROOT / "src").is_dir() and (REPO_ROOT.parent / "src").is_dir():
        REPO_ROOT = REPO_ROOT.parent

# Ensure REPO_ROOT takes top precedence in sys.path
repo_path_str = str(REPO_ROOT)
if repo_path_str in sys.path:
    sys.path.remove(repo_path_str)
sys.path.insert(0, repo_path_str)

# Invalidate import caches and clear cached src modules to prevent stale imports
importlib.invalidate_caches()
for module_name in list(sys.modules):
    if module_name == "src" or module_name.startswith("src."):
        del sys.modules[module_name]

SRC_DIR = REPO_ROOT / "src"
if not SRC_DIR.is_dir():
    raise FileNotFoundError(f"src directory not found at {SRC_DIR}")

print("Repository:", REPO_ROOT)
print("Source modules:", SRC_DIR)

Repository: E:\College\Third Year\Data Science\Course Project\Project Repo
Source modules: E:\College\Third Year\Data Science\Course Project\Project Repo\src


In [2]:
# Install the packages required by the notebook and shared preprocessing modules.
%pip install -q numpy pandas scikit-learn matplotlib jupyter


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 26.2.1
[notice] To update, run: C:\Users\Admin\AppData\Local\Programs\Python\Python310\python.exe -m pip install --upgrade pip


# PadChest preprocessing

This notebook prepares PadChest as a separate metadata-only federated client for the target disease (`Pleural Effusion`).
PadChest uses a rich radiological vocabulary in serialized list format in the `Labels` column.

**Label Policy:**
- `Positive` (`target = 1`): Explicitly documented `pleural effusion` in `Labels`.
- `Negative` (`target = 0`): Radiologically confirmed `normal` studies (`Labels == ['normal']`).
- `Excluded`: Ambiguous cases, other pathologies (e.g. isolated blunting, pneumonia, COPD, cardiomegaly), or missing labels.

**Pipeline Scope:**
- Operates on the full 160K metadata release (`PADCHEST_chest_x_ray_images_labels_160K_01.02.19.csv`) or validation sample (`chest_x_ray_images_labels_sample.csv`).
- Produces the standardized 17-column federated schema compatible with NIH and CheXpert clients without baking machine-specific absolute file paths into CSVs.

In [3]:
import json, platform
from pathlib import Path
import numpy as np
import pandas as pd

from src.config import TARGET_DISEASE, RANDOM_SEED, MAX_POSITIVE_SAMPLES, MAX_NEGATIVE_SAMPLES
from src.dataset_utils import padchest_clean_binary_labels, padchest_find_label_vocabulary, standardize_padchest
from src.preprocessing_utils import (
    assert_no_leakage,
    assert_no_study_leakage,
    build_summary,
    normalize_label,
    parse_label_list,
    require_file,
    sample_clean_cases,
    save_splits,
    save_summary,
    set_reproducibility,
    split_by_patient,
)
set_reproducibility(RANDOM_SEED)
print('Python:', platform.python_version(), 'pandas:', pd.__version__)

# Dataset-specific Google Drive configuration.
BASE_DRIVE_PATH = Path("/content/drive/MyDrive/Healthcare_FL")
if not BASE_DRIVE_PATH.exists() and (REPO_ROOT / "data").exists():
    BASE_DRIVE_PATH = REPO_ROOT

PADCHEST_ROOT = BASE_DRIVE_PATH / "data" / "raw" / "PadChest" if (BASE_DRIVE_PATH / "data" / "raw" / "PadChest").exists() else BASE_DRIVE_PATH / "raw" / "PadChest"
PADCHEST_METADATA_DIR = PADCHEST_ROOT / "metadata" if (PADCHEST_ROOT / "metadata").exists() else PADCHEST_ROOT

FULL_CSV = PADCHEST_METADATA_DIR / "PADCHEST_chest_x_ray_images_labels_160K_01.02.19.csv"
SAMPLE_CSV = PADCHEST_METADATA_DIR / "chest_x_ray_images_labels_sample.csv"

# Full 160K metadata takes top priority; fallback to sample if full is not present
if FULL_CSV.exists():
    METADATA_CSV = FULL_CSV
    IS_SAMPLE = False
elif SAMPLE_CSV.exists():
    METADATA_CSV = SAMPLE_CSV
    IS_SAMPLE = True
else:
    METADATA_CSV = FULL_CSV
    IS_SAMPLE = False

IMAGE_DIR = PADCHEST_ROOT / "images"
OUTPUT_DIR = BASE_DRIVE_PATH / "data" / "processed" / "PadChest" if (BASE_DRIVE_PATH / "data").exists() else BASE_DRIVE_PATH / "processed" / "PadChest"

metadata_path = require_file(METADATA_CSV, "PadChest metadata CSV")

print("Repository:", REPO_ROOT)
print("Google Drive:", BASE_DRIVE_PATH)
print("Dataset:", PADCHEST_ROOT)
print("Metadata:", METADATA_CSV)
print("Is Sample Metadata:", IS_SAMPLE)
print("Output Directory:", OUTPUT_DIR)
print("Required PadChest metadata file exists.")

Python: 3.10.11 pandas: 2.3.3
Repository: E:\College\Third Year\Data Science\Course Project\Project Repo
Google Drive: E:\College\Third Year\Data Science\Course Project\Project Repo
Dataset: E:\College\Third Year\Data Science\Course Project\Project Repo\data\raw\PadChest
Metadata: E:\College\Third Year\Data Science\Course Project\Project Repo\data\raw\PadChest\metadata\PADCHEST_chest_x_ray_images_labels_160K_01.02.19.csv
Is Sample Metadata: False
Output Directory: E:\College\Third Year\Data Science\Course Project\Project Repo\data\processed\PadChest
Required PadChest metadata file exists.


## Load and inspect

We inspect the PadChest metadata, examine label distributions, verify vocabulary matches for `Pleural Effusion`, and filter clean binary cases.

In [4]:
padchest = pd.read_csv(metadata_path, low_memory=False)
print('total raw images/rows:', len(padchest))
print('unique patients:', padchest['PatientID'].nunique() if 'PatientID' in padchest else 'column unavailable')
print('unique studies:', padchest['StudyID'].nunique() if 'StudyID' in padchest else 'column unavailable')
print('unique ImageIDs:', padchest['ImageID'].nunique() if 'ImageID' in padchest else 'column unavailable')
print('columns:', padchest.columns.tolist())

display(padchest.isna().sum().sort_values(ascending=False).head(20).to_frame('missing'))

for col in ['ViewPosition_DICOM', 'Projection', 'Modality_DICOM', 'Manufacturer_DICOM', 'PatientSex_DICOM']:
    if col in padchest.columns:
        display(padchest[col].value_counts(dropna=False).head(10).to_frame(col))

if 'Labels' not in padchest.columns:
    raise KeyError("PadChest metadata must contain a 'Labels' column.")

vocabulary = padchest_find_label_vocabulary(padchest)
matches = [label for label in vocabulary if 'pleural' in normalize_label(label) or 'effusion' in normalize_label(label)]
print('Vocabulary entries matching pleural/effusion:', matches)

# In PadChest vocabulary, 'pleural effusion' is the explicit target finding
TARGET_LABEL = 'pleural effusion'
if normalize_label(TARGET_LABEL) not in [normalize_label(m) for m in matches]:
    print(f'Warning: {TARGET_LABEL!r} not found in current metadata subset vocabulary.')

clean, counts = padchest_clean_binary_labels(
    padchest,
    target_label=TARGET_LABEL,
    negative_label='normal',
)
print('Counts before filtering:', counts)
print('Clean rows:', len(clean))
display(clean[['ImageID', 'PatientID', 'StudyID', 'Labels', 'target']].head(10))

total raw images/rows: 160861
unique patients: 67625
unique studies: 109931
unique ImageIDs: 160861
columns: ['Unnamed: 0', 'ImageID', 'ImageDir', 'StudyDate_DICOM', 'StudyID', 'PatientID', 'PatientBirth', 'PatientSex_DICOM', 'ViewPosition_DICOM', 'Projection', 'MethodProjection', 'Pediatric', 'Modality_DICOM', 'Manufacturer_DICOM', 'PhotometricInterpretation_DICOM', 'PixelRepresentation_DICOM', 'PixelAspectRatio_DICOM', 'SpatialResolution_DICOM', 'BitsStored_DICOM', 'WindowCenter_DICOM', 'WindowWidth_DICOM', 'Rows_DICOM', 'Columns_DICOM', 'XRayTubeCurrent_DICOM', 'Exposure_DICOM', 'ExposureInuAs_DICOM', 'ExposureTime', 'RelativeXRayExposure_DICOM', 'ReportID', 'Report', 'MethodLabel', 'Labels', 'Localizations', 'LabelsLocalizationsBySentence', 'labelCUIS', 'LocalizationsCUIS']


,missing
PixelAspectRatio_DICOM,112858
SpatialResolution_DICOM,96593
ExposureInuAs_DICOM,70573
ViewPosition_DICOM,69085
XRayTubeCurrent_DICOM,57985
Exposure_DICOM,47417
ExposureTime,47417
RelativeXRayExposure_DICOM,21454
WindowWidth_DICOM,1223
WindowCenter_DICOM,1223


,ViewPosition_DICOM
ViewPosition_DICOM,
NaN,69085
POSTEROANTERIOR,54869
LATERAL,24527
PA,8506
LL,3383
ANTEROPOSTERIOR,369
AP,104
OBLICUA,11
RL,5


,Projection
Projection,
PA,91728
L,49579
AP_horizontal,14346
AP,4559
COSTAL,630
EXCLUDE,11
UNK,8


,Modality_DICOM
Modality_DICOM,
CR,148862
DX,11999


,Manufacturer_DICOM
Manufacturer_DICOM,
PhilipsMedicalSystems,81084
ImagingDynamicsCompanyLtd,79777


,PatientSex_DICOM
PatientSex_DICOM,
M,80920
F,79919
O,18
NaN,4


Vocabulary entries matching pleural/effusion: ['apical pleural thickening', 'calcified pleural plaques', 'calcified pleural thickening', 'loculated fissural effusion', 'loculated pleural effusion', 'pericardial effusion', 'pleural effusion', 'pleural mass', 'pleural plaques', 'pleural thickening']


Counts before filtering: {'positive_before_filter': 9853, 'negative_before_filter': 50390, 'excluded_ambiguous_before_filter': 100618}
Clean rows: 60243


,ImageID,PatientID,StudyID,Labels,target
0,20536686640136348236148679891455886468_k6ga29.png,839860488694292331637988235681460987,20536686640136348236148679891455886468,['normal'],0
6,3137231742710829928-247610802266403640553_km0q...,93535126770783451980359712286922420997,3137231742710829928-247610802266403640553,"['laminar atelectasis', 'pleural effusion', 'a...",1
7,3137231742710829928-247610802266403640553_kine...,93535126770783451980359712286922420997,3137231742710829928-247610802266403640553,"['laminar atelectasis', 'pleural effusion', 'a...",1
9,313723174271082992847610802266403640553-2_40kx...,93535126770783451980359712286922420997,313723174271082992847610802266403640553-2,"['alveolar pattern', 'pleural effusion', 'endo...",1
10,313723174271082992847610802266403640553_w8dk8c...,93535126770783451980359712286922420997,313723174271082992847610802266403640553,"['pleural effusion', 'infiltrates', 'endotrach...",1
11,313723174271082992847610802266403640553_5_2icw...,93535126770783451980359712286922420997,313723174271082992847610802266403640553_5,"['heart insufficiency', 'pleural effusion', 's...",1
14,238285621348398466668514178112618553012_a7k6dv...,76478060312810134599679306720116852498,238285621348398466668514178112618553012,['normal'],0
15,152191969602076825998375638267191596461_ck9qkz...,211676018114322936525876155155978326212,152191969602076825998375638267191596461,['normal'],0
20,84289138814897824414670894017743137660_xjsw7k.png,132905598244344199302356904572008903408,84289138814897824414670894017743137660,['normal'],0
23,258055173326723517262722952856353517225_hcez67...,163995406284018596913093962224045518908,258055173326723517262722952856353517225,['normal'],0


In [5]:
# Standardize metadata to the 17-column federated schema
# Paths are kept reproducible and relative without machine-specific absolute prefixes
standardized = standardize_padchest(clean)
print('Standardized schema shape:', standardized.shape)
print('Columns match 17-column standard:', len(standardized.columns) == 17)
display(standardized.head())

# Optional sample filtering
standardized = sample_clean_cases(standardized, MAX_POSITIVE_SAMPLES, MAX_NEGATIVE_SAMPLES, RANDOM_SEED)

# Patient-level split: all images from the same patient stay in the same split
splits = split_by_patient(standardized, seed=RANDOM_SEED, train_size=0.8, val_size=0.1)

# Strict patient and study leakage assertions
assert_no_leakage(splits)
assert_no_study_leakage(splits)
print('patient and study leakage checks: passed')

for split_name, split_frame in splits.items():
    print(
        f'{split_name}: {len(split_frame)} images | '
        f'{split_frame["patient_id"].nunique()} patients | '
        f'{split_frame["study_id"].nunique()} studies | '
        f'positive: {int((split_frame["target"] == 1).sum())} | '
        f'negative: {int((split_frame["target"] == 0).sum())}'
    )

display(pd.DataFrame([
    {
        'split': split_name,
        'images': len(split_frame),
        'patients': split_frame['patient_id'].nunique(),
        'studies': split_frame['study_id'].nunique(),
        'positive': int((split_frame['target'] == 1).sum()),
        'negative': int((split_frame['target'] == 0).sum()),
    }
    for split_name, split_frame in splits.items()
]))

save_splits(splits, OUTPUT_DIR)

summary = build_summary(
    splits,
    'PadChest',
    {
        'target_disease': TARGET_DISEASE,
        'source_target_label': TARGET_LABEL,
        'negative_label': 'normal',
        'random_seed': RANDOM_SEED,
        'max_positive_samples': MAX_POSITIVE_SAMPLES,
        'max_negative_samples': MAX_NEGATIVE_SAMPLES,
        'filter_counts_before_filtering': counts,
        'is_sample_metadata': IS_SAMPLE,
        'split_strategy': 'patient_level_80_10_10',
        'image_paths_materialized': False,
    },
)
save_summary(summary, OUTPUT_DIR)
print(json.dumps(summary, indent=2))
print('Saved files:', sorted([path.name for path in OUTPUT_DIR.glob('*')]))

Standardized schema shape: (60243, 17)
Columns match 17-column standard: True


,image_id,patient_id,study_id,source,target,age,sex,view_position,projection,image_width,image_height,pixel_spacing_x,pixel_spacing_y,scanner_manufacturer,image_path,finding_labels,follow_up
0,20536686640136348236148679891455886468_k6ga29.png,839860488694292331637988235681460987,20536686640136348236148679891455886468,PadChest,0,84.0,F,POSTEROANTERIOR,PA,1872,1728,NaN,NaN,ImagingDynamicsCompanyLtd,PadChest/images/0/2053668664013634823614867989...,['normal'],NaN
6,3137231742710829928-247610802266403640553_km0q...,93535126770783451980359712286922420997,3137231742710829928-247610802266403640553,PadChest,1,58.0,M,NaN,L,2976,2965,NaN,NaN,PhilipsMedicalSystems,PadChest/images/0/3137231742710829928-24761080...,"['laminar atelectasis', 'pleural effusion', 'a...",NaN
7,3137231742710829928-247610802266403640553_kine...,93535126770783451980359712286922420997,3137231742710829928-247610802266403640553,PadChest,1,58.0,M,NaN,PA,2973,2968,NaN,NaN,PhilipsMedicalSystems,PadChest/images/0/3137231742710829928-24761080...,"['laminar atelectasis', 'pleural effusion', 'a...",NaN
9,313723174271082992847610802266403640553-2_40kx...,93535126770783451980359712286922420997,313723174271082992847610802266403640553-2,PadChest,1,58.0,M,NaN,AP_horizontal,2140,1760,NaN,NaN,PhilipsMedicalSystems,PadChest/images/0/3137231742710829928476108022...,"['alveolar pattern', 'pleural effusion', 'endo...",NaN
10,313723174271082992847610802266403640553_w8dk8c...,93535126770783451980359712286922420997,313723174271082992847610802266403640553,PadChest,1,58.0,M,NaN,AP_horizontal,2140,1760,NaN,NaN,PhilipsMedicalSystems,PadChest/images/0/3137231742710829928476108022...,"['pleural effusion', 'infiltrates', 'endotrach...",NaN


patient and study leakage checks: passed
train: 48340 images | 26729 patients | 34092 studies | positive: 7934 | negative: 40406
val: 5977 images | 3341 patients | 4246 studies | positive: 965 | negative: 5012
test: 5926 images | 3342 patients | 4229 studies | positive: 954 | negative: 4972


,split,images,patients,studies,positive,negative
0,train,48340,26729,34092,7934,40406
1,val,5977,3341,4246,965,5012
2,test,5926,3342,4229,954,4972


{
  "source": "PadChest",
  "parameters": {
    "target_disease": "Pleural Effusion",
    "source_target_label": "pleural effusion",
    "negative_label": "normal",
    "random_seed": 42,
    "max_positive_samples": null,
    "max_negative_samples": null,
    "filter_counts_before_filtering": {
      "positive_before_filter": 9853,
      "negative_before_filter": 50390,
      "excluded_ambiguous_before_filter": 100618
    },
    "is_sample_metadata": false,
    "split_strategy": "patient_level_80_10_10",
    "image_paths_materialized": false
  },
  "splits": {
    "train": {
      "images": 48340,
      "patients": 26729,
      "studies": 34092,
      "positive": 7934,
      "negative": 40406
    },
    "val": {
      "images": 5977,
      "patients": 3341,
      "studies": 4246,
      "positive": 965,
      "negative": 5012
    },
    "test": {
      "images": 5926,
      "patients": 3342,
      "studies": 4229,
      "positive": 954,
      "negative": 4972
    }
  },
  "total_images"